# Face Anti-Spoofing Detection for Authentication Systems
## Project STT: 12 | Architecture: ResNet34 + Local Binary Patterns (LBP) Texture Fusion
### Google Colab GPU Training & Evaluation Notebook

This notebook provides the complete end-to-end training and evaluation pipeline on Google Colab (T4 / V100 / A100 GPU).

---

### Step 1: Check GPU Environment and Hardware Acceleration
Ensure runtime type is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated:{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Step 2: Clone or Mount Project Files
Option A: Mount Google Drive if you stored the project folder there.
Option B: Upload the project zip directly to Colab.

In [ ]:
# Mount Google Drive (Optional)
from google.colab import drive
drive.mount('/content/drive')

# Change directory to project root
# %cd /content/drive/MyDrive/face_antispoofing_stt12

### Step 3: Install Required Dependencies
Install scientific libraries and tools matching Python 3.10-3.12.

In [ ]:
!pip install -q pyyaml opencv-python scikit-learn scikit-image tqdm matplotlib seaborn pandas

### Step 4: Verify Environment and Preprocessed Dataset
> **Important:** Because raw MSU-MFSD is large (~10.5 GB), we preprocess the video frames on PC first using `python prepare_dataset.py`, and only upload the lightweight `processed/` folder (or `processed.zip`) to Colab.

In [ ]:
# If you uploaded processed.zip, unzip it here:
# !unzip -q processed.zip -d processed

import pandas as pd
from pathlib import Path

meta_path = Path("processed/metadata.csv")
if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"Total Preprocessed Frames: {len(df)}")
    print("\nDistribution by Split:")
    print(df.groupby(['split', 'label_name']).size().unstack(fill_value=0))
else:
    print("Warning: processed/metadata.csv not found. Please upload the processed folder.")

### Step 5: Train ResNet34 + LBP Feature Fusion Model (Proposed)
Trains the dual-stream multimodal network with staged fine-tuning and AdamW optimizer.

In [ ]:
!python train.py --model fusion --epochs 20 --batch-size 16 --lr 0.0001

### Step 6: Train ResNet34 Baseline Model (Comparative Benchmark)
Trains pure ResNet34 without texture descriptors under the identical training protocol.

In [ ]:
!python train.py --model baseline --epochs 20 --batch-size 16 --lr 0.0001

### Step 7: Evaluate Models on Independent Test Set
Computes Accuracy, Precision, Recall, F1, APCER, BPCER, ACER, AUC, and Video-Level Aggregations.

In [ ]:
# Evaluate Proposed Fusion Architecture
!python evaluate.py --model fusion --checkpoint checkpoints/best_fusion_model.pth --split test

# Evaluate Baseline Architecture
!python evaluate.py --model baseline --checkpoint checkpoints/best_baseline_model.pth --split test

### Step 8: Generate Comparative Visualizations and Reports
Generates side-by-side comparison tables, bar charts, confusion matrices, and ROC curves.

In [ ]:
!python evaluate.py --compare --baseline-ckpt checkpoints/best_baseline_model.pth --checkpoint checkpoints/best_fusion_model.pth

from IPython.display import Image, display
print("=== Model Comparison Chart ===")
display(Image("results/model_comparison.png"))

print("\n=== Fusion Confusion Matrix ===")
display(Image("results/confusion_matrix_fusion_test.png"))

print("\n=== Fusion ROC Curve ===")
display(Image("results/roc_curve_fusion_test.png"))

### Step 9: Download Trained Weights to Local Windows Machine
Download `best_fusion_model.pth` and place it in `checkpoints/` on your Windows PC to run the realtime webcam demo.

In [ ]:
from google.colab import files
files.download('checkpoints/best_fusion_model.pth')
files.download('results/model_comparison.png')